# Audience Intelligence Pipeline — Databricks PySpark Demo

This notebook demonstrates a role-relevant audience intelligence pipeline for a sports and entertainment context.

It shows how to:
- ingest ticketing, web, email, and subscription data
- build a fan 360 model for segmentation
- write Delta outputs with partitioning
- apply incremental CDC-style updates with MERGE
- validate data quality and prepare the job for scheduled runs

## Scenario and prerequisites

Scenario: unify fan interaction data across ticket purchases, website behavior, email engagement, and subscriptions into a single audience model that product and marketing teams can use.

Prerequisites:
- Databricks Runtime with Delta Lake support (11+ recommended)
- a small cluster or SQL warehouse for notebook execution
- DBFS or mounted object storage path for Delta output
- optional: adapt sample inputs to CSV / Parquet files from real or mock source systems

In [ ]:
# Imports and basic Spark configuration
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
try:
    from delta.tables import DeltaTable
except Exception:
    DeltaTable = None
    print('DeltaTable not available — run on Databricks runtime for full Delta support')

# Tune the demo for smaller clusters
spark.conf.set('spark.sql.shuffle.partitions', '8')

## 1 — Create sample audience data

The demo uses four source domains that resemble a simplified audience intelligence platform:
- ticket purchases
- website sessions
- email campaign engagement
- subscriptions / memberships

In [ ]:
# Sample ticket purchases
ticket_purchases = [
    (101, 'Lions FC', '2026-03-10', 2, 180.0),
    (101, 'Lions FC', '2026-03-28', 1, 95.0),
    (102, 'Lions FC', '2026-03-14', 4, 320.0),
    (103, 'Tigers FC', '2026-03-16', 1, 70.0),
    (104, 'Lions FC', '2026-03-22', 2, 160.0)
 ]
ticket_schema = StructType([
    StructField('fan_id', IntegerType(), False),
    StructField('club', StringType(), True),
    StructField('purchase_date', StringType(), True),
    StructField('tickets', IntegerType(), True),
    StructField('ticket_revenue', DoubleType(), True)
 ])
ticket_df = spark.createDataFrame(ticket_purchases, schema=ticket_schema).withColumn('purchase_date', F.to_date('purchase_date'))

# Sample website sessions
web_sessions = [
    (101, '2026-03-27', 5, 22),
    (102, '2026-03-28', 8, 31),
    (103, '2026-03-20', 2, 6),
    (104, '2026-03-29', 6, 19),
    (105, '2026-03-30', 4, 14)
 ]
web_schema = StructType([
    StructField('fan_id', IntegerType(), False),
    StructField('session_date', StringType(), True),
    StructField('sessions', IntegerType(), True),
    StructField('page_views', IntegerType(), True)
 ])
web_df = spark.createDataFrame(web_sessions, schema=web_schema).withColumn('session_date', F.to_date('session_date'))

# Sample email engagement
email_events = [
    (101, 'season_launch', 5, 3, 1),
    (102, 'season_launch', 4, 1, 0),
    (103, 'renewal_push', 3, 2, 0),
    (104, 'vip_offer', 6, 5, 2),
    (105, 'welcome_series', 2, 1, 1)
 ]
email_schema = StructType([
    StructField('fan_id', IntegerType(), False),
    StructField('campaign_name', StringType(), True),
    StructField('emails_sent', IntegerType(), True),
    StructField('emails_opened', IntegerType(), True),
    StructField('emails_clicked', IntegerType(), True)
 ])
email_df = spark.createDataFrame(email_events, schema=email_schema)

# Sample subscriptions / memberships
subscriptions = [
    (101, 'member', 'active', '2026-12-31', 199.0),
    (102, 'member', 'active', '2026-10-31', 149.0),
    (103, 'newsletter', 'inactive', None, 0.0),
    (104, 'member', 'active', '2027-01-15', 249.0),
    (105, 'newsletter', 'active', None, 0.0)
 ]
subscription_schema = StructType([
    StructField('fan_id', IntegerType(), False),
    StructField('subscription_type', StringType(), True),
    StructField('subscription_status', StringType(), True),
    StructField('renewal_date', StringType(), True),
    StructField('subscription_value', DoubleType(), True)
 ])
subscription_df = spark.createDataFrame(subscriptions, schema=subscription_schema).withColumn('renewal_date', F.to_date('renewal_date'))

display(ticket_df)
display(web_df)
display(email_df)
display(subscription_df)

## 2 — Build the fan 360 audience model

Aggregate each source to the fan level, then combine them into a single modeled dataset that can be used for segmentation and activation.

In [ ]:
ticket_agg_df = ticket_df.groupBy('fan_id').agg(
    F.first('club', ignorenulls=True).alias('favorite_club'),
    F.sum('tickets').alias('tickets_purchased'),
    F.sum('ticket_revenue').alias('ticket_ltv'),
    F.max('purchase_date').alias('last_ticket_purchase')
)

web_agg_df = web_df.groupBy('fan_id').agg(
    F.sum('sessions').alias('total_sessions'),
    F.sum('page_views').alias('total_page_views'),
    F.max('session_date').alias('last_session_date')
)

email_agg_df = email_df.groupBy('fan_id').agg(
    F.sum('emails_sent').alias('emails_sent'),
    F.sum('emails_opened').alias('emails_opened'),
    F.sum('emails_clicked').alias('emails_clicked')
).withColumn(
    'email_open_rate',
    F.when(F.col('emails_sent') == 0, F.lit(0.0)).otherwise(F.col('emails_opened') / F.col('emails_sent'))
)

fan_360_df = ticket_agg_df.join(web_agg_df, 'fan_id', 'full') \
    .join(email_agg_df, 'fan_id', 'full') \
    .join(subscription_df, 'fan_id', 'full') \
    .fillna({
        'favorite_club': 'Unknown',
        'tickets_purchased': 0,
        'ticket_ltv': 0.0,
        'total_sessions': 0,
        'total_page_views': 0,
        'emails_sent': 0,
        'emails_opened': 0,
        'emails_clicked': 0,
        'email_open_rate': 0.0,
        'subscription_status': 'unknown',
        'subscription_value': 0.0
    }) \
    .withColumn('total_value', F.col('ticket_ltv') + F.col('subscription_value')) \
    .withColumn(
        'engagement_segment',
        F.when((F.col('total_value') >= 350) & (F.col('email_open_rate') >= 0.50), 'high_value_engaged')
         .when((F.col('tickets_purchased') >= 2) | (F.col('total_sessions') >= 5), 'active_fan')
         .when(F.col('email_open_rate') > 0, 'reachable_low_value')
         .otherwise('at_risk')
    )

display(fan_360_df.orderBy('fan_id'))

In [ ]:
# Write the modeled audience output to Delta
target_path = '/tmp/yash_demo/fan_360_delta'
fan_360_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').partitionBy('engagement_segment').save(target_path)

read_back = spark.read.format('delta').load(target_path)
display(read_back.orderBy('fan_id'))

# Example downstream summary for marketers / product managers
segment_summary_df = read_back.groupBy('engagement_segment').agg(
    F.count('*').alias('fans'),
    F.round(F.sum('total_value'), 2).alias('segment_value'),
    F.round(F.avg('email_open_rate'), 3).alias('avg_open_rate')
)
display(segment_summary_df.orderBy(F.desc('segment_value')))

## 3 — Apply incremental updates with MERGE

Simulate a daily refresh where new purchases and subscription changes arrive, then update the modeled fan 360 table using a Delta MERGE.

In [ ]:
incremental_ticket_updates = [
    (101, 'Lions FC', '2026-04-02', 1, 110.0),
    (105, 'Tigers FC', '2026-04-03', 2, 140.0)
 ]
incremental_ticket_df = spark.createDataFrame(incremental_ticket_updates, schema=ticket_schema).withColumn('purchase_date', F.to_date('purchase_date'))

updated_ticket_df = ticket_df.unionByName(incremental_ticket_df)

updated_ticket_agg_df = updated_ticket_df.groupBy('fan_id').agg(
    F.first('club', ignorenulls=True).alias('favorite_club'),
    F.sum('tickets').alias('tickets_purchased'),
    F.sum('ticket_revenue').alias('ticket_ltv'),
    F.max('purchase_date').alias('last_ticket_purchase')
)

incremental_subscription_updates = [
    (105, 'member', 'active', '2027-03-31', 179.0)
 ]
incremental_subscription_df = spark.createDataFrame(incremental_subscription_updates, schema=subscription_schema).withColumn('renewal_date', F.to_date('renewal_date'))

updated_subscription_df = subscription_df.unionByName(incremental_subscription_df) \
    .withColumn('row_priority', F.when(F.col('subscription_type') == 'member', 1).otherwise(0))

subscription_window = Window.partitionBy('fan_id').orderBy(F.desc('row_priority'), F.desc('renewal_date'))
latest_subscription_df = updated_subscription_df \
    .withColumn('rn', F.row_number().over(subscription_window)) \
    .filter(F.col('rn') == 1) \
    .drop('rn', 'row_priority')

updated_fan_360_df = updated_ticket_agg_df.join(web_agg_df, 'fan_id', 'full') \
    .join(email_agg_df, 'fan_id', 'full') \
    .join(latest_subscription_df, 'fan_id', 'full') \
    .fillna({
        'favorite_club': 'Unknown',
        'tickets_purchased': 0,
        'ticket_ltv': 0.0,
        'total_sessions': 0,
        'total_page_views': 0,
        'emails_sent': 0,
        'emails_opened': 0,
        'emails_clicked': 0,
        'email_open_rate': 0.0,
        'subscription_status': 'unknown',
        'subscription_value': 0.0
    }) \
    .withColumn('total_value', F.col('ticket_ltv') + F.col('subscription_value')) \
    .withColumn(
        'engagement_segment',
        F.when((F.col('total_value') >= 350) & (F.col('email_open_rate') >= 0.50), 'high_value_engaged')
         .when((F.col('tickets_purchased') >= 2) | (F.col('total_sessions') >= 5), 'active_fan')
         .when(F.col('email_open_rate') > 0, 'reachable_low_value')
         .otherwise('at_risk')
    )

merge_columns = {
    'favorite_club': 's.favorite_club',
    'tickets_purchased': 's.tickets_purchased',
    'ticket_ltv': 's.ticket_ltv',
    'total_sessions': 's.total_sessions',
    'total_page_views': 's.total_page_views',
    'emails_sent': 's.emails_sent',
    'emails_opened': 's.emails_opened',
    'emails_clicked': 's.emails_clicked',
    'email_open_rate': 's.email_open_rate',
    'subscription_type': 's.subscription_type',
    'subscription_status': 's.subscription_status',
    'renewal_date': 's.renewal_date',
    'subscription_value': 's.subscription_value',
    'last_ticket_purchase': 's.last_ticket_purchase',
    'last_session_date': 's.last_session_date',
    'total_value': 's.total_value',
    'engagement_segment': 's.engagement_segment'
}

if DeltaTable is not None and DeltaTable.isDeltaTable(spark, target_path):
    delta_table = DeltaTable.forPath(spark, target_path)
    (
        delta_table.alias('t')
        .merge(updated_fan_360_df.alias('s'), 't.fan_id = s.fan_id')
        .whenMatchedUpdate(set=merge_columns)
        .whenNotMatchedInsertAll()
        .execute()
    )

display(spark.read.format('delta').load(target_path).orderBy('fan_id'))

## 4 — Validate audience-model quality

Basic checks here focus on the kinds of guarantees an activation-ready fan model should have: one row per fan, no null identifiers, and reasonable metric values.

In [ ]:
modeled_df = spark.read.format('delta').load(target_path)

if modeled_df.filter(F.col('fan_id').isNull()).count() > 0:
    raise ValueError('fan_id should never be null')

duplicate_fans = modeled_df.groupBy('fan_id').count().filter(F.col('count') > 1).count()
if duplicate_fans > 0:
    raise ValueError('fan_id should be unique in the fan 360 model')

negative_values = modeled_df.filter((F.col('ticket_ltv') < 0) | (F.col('subscription_value') < 0)).count()
if negative_values > 0:
    raise ValueError('monetary values should not be negative')

print('Modeled rows:', modeled_df.count())
display(modeled_df.groupBy('engagement_segment').count().orderBy(F.desc('count')))

## 5 — Performance and production notes

For a larger production pipeline, typical tuning levers would include:
- repartitioning on high-cardinality processing keys only when it improves shuffle behavior
- partitioning Delta outputs by a commonly filtered business dimension such as segment or club
- broadcasting genuinely small lookup tables
- monitoring skew, shuffle volume, and file sizes in Spark UI
- adding Z-ORDER / OPTIMIZE where Delta table access patterns justify it

In [ ]:
optimized_df = modeled_df.repartition('engagement_segment').cache()
optimized_df.count()

club_value_df = optimized_df.groupBy('favorite_club').agg(
    F.round(F.sum('total_value'), 2).alias('club_value'),
    F.count('*').alias('fans')
)
display(club_value_df.orderBy(F.desc('club_value')))

## 6 — Parameterize for scheduled runs

In Databricks Jobs, parameterization makes the notebook reusable across brands, clubs, or run dates. This is also the bridge to ADF or any external orchestrator.

In [ ]:
try:
    dbutils.widgets.text('brand_name', 'Two Circles Demo', 'Brand name')
    dbutils.widgets.text('run_date', '2026-04-09', 'Run date')
    dbutils.widgets.text('target_path', target_path, 'Target Delta path')
    brand_name = dbutils.widgets.get('brand_name')
    run_date = dbutils.widgets.get('run_date')
    target_path_param = dbutils.widgets.get('target_path')
except NameError:
    print('dbutils not available in this environment; using defaults from the notebook')
    brand_name = 'Two Circles Demo'
    run_date = '2026-04-09'
    target_path_param = target_path

print('brand_name =', brand_name)
print('run_date =', run_date)
print('target_path =', target_path_param)

## Next steps

- Replace inline sample data with CSV or Parquet inputs that resemble real ticketing, web, campaign, and subscription systems.
- Split the pipeline into bronze, silver, and gold layers for a clearer lakehouse story.
- Schedule the notebook via Databricks Jobs and optionally trigger it from ADF or CI/CD.
- Add one downstream dashboard, activation export, or stakeholder-facing metric table.

This notebook now maps directly to the Two Circles role because it demonstrates:
- audience-centric data modeling
- incremental and CDC-style processing
- Delta Lake write and merge patterns
- data-quality validation
- product and marketing-oriented outputs